# Lab 5 — Hyperspectral data & water quality analysis

This notebook completes **Tasks 3–4** from `TODO.pdf`:

1. False-colour composites from the airborne hyperspectral cube
2. Water-quality proxies: Chl-a (NDCI), DOC, turbidity
3. Sentinel-2 download for the closest date to the airborne acquisition
4. Comparison of indices between airborne and Sentinel-2
5. SAM classification (Lab 3 style) with Sentinel-2 calibration from airborne match-ups

> **Data:** place the ENVI `.hdr` + `.bsq` pair in `data/images/`.  
> If no data is present, run `python generate_test_cube.py` first (small synthetic cube).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform_bounds
from rasterio.crs import CRS

import pystac_client
import planetary_computer

from hs_utils import (
    find_hdr_files,
    load_envi,
    parse_wavelengths,
    get_ignore_value,
    get_reflectance_scale,
    parse_acquisition_date,
    mask_invalid,
    to_reflectance,
    false_color_composite,
    compute_ndci,
    compute_doc_proxy,
    compute_turbidity_proxy,
    compute_s2_ndci,
    compute_s2_doc_proxy,
    compute_s2_turbidity_proxy,
    convolve_cube_to_s2,
    fit_linear_calibration,
    apply_calibration,
    spectral_angle,
    load_spectral_library,
    S2_WAVELENGTHS_NM,
)

LAB_DIR = Path('.').resolve()
DATA_DIR = LAB_DIR / 'data' / 'images'
LIBRARY_DIR = LAB_DIR / 'data' / 'spectral_library'
S2_DIR = LAB_DIR / 'data' / 'downloads' / 'sentinel2'
S2_DIR.mkdir(parents=True, exist_ok=True)

print('Lab directory:', LAB_DIR)

In [ ]:
# Auto-generate a small test cube if no real data is available
hdrs = find_hdr_files(DATA_DIR)
if not hdrs:
    print('No ENVI header found — generating synthetic test cube...')
    import subprocess
    subprocess.run(['python', 'generate_test_cube.py'], check=True, cwd=LAB_DIR)
    hdrs = find_hdr_files(DATA_DIR)

HDR_PATH = hdrs[0]
print('Using:', HDR_PATH.name)

In [ ]:
# Load hyperspectral cube (memory-mapped — only reads bands on demand)
img = load_envi(HDR_PATH)
meta = img.metadata
wavelengths = parse_wavelengths(meta)
ignore_value = get_ignore_value(meta)
scale = get_reflectance_scale(meta)
acq_date = parse_acquisition_date(meta, HDR_PATH)

print(f'Scene: {img.nrows} x {img.ncols} x {img.nbands} bands')
print('Acquisition date:', acq_date)
print('Reflectance scale:', scale)

# Read full cube for index mapping (OK for test data; for multi-GB scenes use band-wise reads)
raw = np.asarray(img.load())
cube = mask_invalid(raw.astype(np.float64), ignore_value)
cube = to_reflectance(cube, scale)

## Task 3a — False-colour composites

In [ ]:
composites = {
    'RGB (default bands)': false_color_composite(cube, wavelengths, (665, 560, 490), ignore_value=None, scale=1.0),
    'CIR (NIR-Red-Green)': false_color_composite(cube, wavelengths, (800, 665, 560), ignore_value=None, scale=1.0),
    'SWIR composite': false_color_composite(cube, wavelengths, (900, 700, 550), ignore_value=None, scale=1.0),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, rgb) in zip(axes, composites.items()):
    ax.imshow(rgb)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Task 3b — Water quality indices (airborne)

In [ ]:
ndci_hs = compute_ndci(cube, wavelengths)
doc_hs = compute_doc_proxy(cube, wavelengths)
turb_hs = compute_turbidity_proxy(cube, wavelengths)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title, cmap in zip(
    axes,
    [ndci_hs, doc_hs, turb_hs],
    ['NDCI (Chl-a proxy)', 'DOC proxy (G/R)', 'Turbidity proxy (NDTI)'],
    ['RdYlGn', 'YlOrBr', 'viridis'],
):
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Task 2 — Spectral library

In [ ]:
# Build library if missing
if not (LIBRARY_DIR / 'manifest.json').exists():
    import subprocess
    subprocess.run(['python', 'build_spectral_library.py', str(HDR_PATH)], check=True, cwd=LAB_DIR)

library = load_spectral_library(LIBRARY_DIR)
for cls, specs in library.items():
    print(f'{cls:12s}: {specs.shape[0]} spectra, {specs.shape[1]} bands')

fig, ax = plt.subplots(figsize=(10, 4))
for cls, specs in library.items():
    ax.plot(wavelengths, np.nanmedian(specs, axis=0), label=cls)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Reflectance')
ax.set_title('Spectral library (median signatures)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Task 3c — Download Sentinel-2 (closest date)

In [ ]:
# AOI from ENVI map info
from hs_utils import bbox_wgs84_from_map_info

BBOX = bbox_wgs84_from_map_info(meta, img.ncols, img.nrows)
if BBOX is None:
    BBOX = [14.0, 52.0, 14.6, 52.6]  # Fallback: Odra estuary

if acq_date:
    try:
        acq_dt = datetime.strptime(acq_date[:10], '%Y-%m-%d')
    except ValueError:
        acq_dt = datetime(2022, 10, 15)
else:
    acq_dt = datetime(2022, 10, 15)

window_start = (acq_dt - timedelta(days=14)).strftime('%Y-%m-%d')
window_end = (acq_dt + timedelta(days=14)).strftime('%Y-%m-%d')
print('Search window:', window_start, '->', window_end)
print('BBOX (WGS84):', [round(v, 4) for v in BBOX])

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
search = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime=f'{window_start}/{window_end}',
    query={'eo:cloud_cover': {'lt': 40}},
)
items = sorted(search.items(), key=lambda it: (
    abs((datetime.fromisoformat(it.properties['datetime'].replace('Z', '+00:00')).replace(tzinfo=None) - acq_dt).days),
    it.properties.get('eo:cloud_cover', 100),
))
print(f'Found {len(items)} Sentinel-2 L2A scenes')
if not items:
    raise RuntimeError('No Sentinel-2 scenes found — widen the date window or check BBOX')

s2_item = planetary_computer.sign(items[0])
s2_date = s2_item.properties['datetime'][:10]
print('Selected scene:', s2_item.id, '| date:', s2_date, '| cloud:', s2_item.properties.get('eo:cloud_cover'))


In [ ]:
# Download key Sentinel-2 bands
BANDS = ['B03', 'B04', 'B05', 'B08', 'B11']
s2_arrays = {}

for bname in BANDS:
    href = s2_item.assets[bname].href
    local = S2_DIR / f'{s2_item.id}_{bname}.tif'
    if not local.exists():
        import urllib.request
        print('Downloading', bname)
        urllib.request.urlretrieve(href, local)
    with rasterio.open(local) as ds:
        s2_arrays[bname] = ds.read(1).astype(np.float32) / 10000.0

from hs_utils import align_raster_stack
s2_arrays = align_raster_stack(s2_arrays, 'B04')
print('Downloaded bands:', list(s2_arrays.keys()), '| shape:', s2_arrays['B04'].shape)

## Task 3d — Sentinel-2 indices & comparison

In [ ]:
b03, b04, b05 = s2_arrays['B03'], s2_arrays['B04'], s2_arrays['B05']
ndci_s2 = compute_s2_ndci(b04, b05)
doc_s2 = compute_s2_doc_proxy(b03, b04)
turb_s2 = compute_s2_turbidity_proxy(b03, b04)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
pairs = [
    (ndci_hs, ndci_s2, 'NDCI'),
    (doc_hs, doc_s2, 'DOC proxy'),
    (turb_hs, turb_s2, 'Turbidity proxy'),
]
for col, (hs, s2, name) in enumerate(pairs):
    axes[0, col].imshow(hs, cmap='RdYlGn')
    axes[0, col].set_title(f'Airborne {name}')
    axes[0, col].axis('off')
    axes[1, col].imshow(s2, cmap='RdYlGn')
    axes[1, col].set_title(f'Sentinel-2 {name} ({s2_date})')
    axes[1, col].axis('off')
plt.tight_layout()
plt.show()

# Convolve airborne cube to S2 bands for pixel-level comparison
hs_s2 = convolve_cube_to_s2(cube, wavelengths)
hs_ndci = compute_s2_ndci(hs_s2['B04'], hs_s2['B05']).ravel()
s2_ndci_flat = ndci_s2.ravel()
n = min(hs_ndci.size, s2_ndci_flat.size)
valid = np.isfinite(hs_ndci[:n]) & np.isfinite(s2_ndci_flat[:n])
if valid.sum() > 100:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(s2_ndci_flat[:n][valid][::50], hs_ndci[:n][valid][::50], s=4, alpha=0.3)
    ax.set_xlabel('Sentinel-2 NDCI')
    ax.set_ylabel('Airborne NDCI (S2-convolved)')
    ax.set_title('Index comparison (subsampled)')
    ax.grid(alpha=0.3)
    plt.show()


## Task 4 — SAM classification & Sentinel-2 calibration

In [ ]:
# Reference spectra: median per class from library
ref_spectra = {cls: np.nanmedian(specs, axis=0) for cls, specs in library.items()}

# SAM map for each class (airborne, full resolution)
lines, samples, bands = cube.shape
flat = cube.reshape(-1, bands)
sam_maps = {}
for cls, ref in ref_spectra.items():
    sam_maps[cls] = spectral_angle(flat, ref).reshape(lines, samples)

fig, axes = plt.subplots(1, len(sam_maps), figsize=(4 * len(sam_maps), 4))
if len(sam_maps) == 1:
    axes = [axes]
for ax, (cls, sam) in zip(axes, sam_maps.items()):
    im = ax.imshow(sam, cmap='jet_r', vmin=0, vmax=np.nanpercentile(sam, 95))
    ax.set_title(f'SAM — {cls}')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

# Winner-takes-all classification (smallest angle)
stack = np.stack([sam_maps[c] for c in sam_maps], axis=-1)
class_names = list(sam_maps.keys())
label_idx = np.argmin(stack, axis=-1)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(label_idx, cmap='tab10', vmin=0, vmax=len(class_names) - 1)
ax.set_title('SAM classification (airborne)')
ax.axis('off')
plt.show()

In [ ]:
# Calibrate Sentinel-2 using airborne spectra convolved to S2 bands
# Linear fit per band: HS_conv = slope * S2 + intercept

calibration = {}
for band in ['B03', 'B04', 'B05', 'B08']:
    hs_band = hs_s2[band].ravel()
    # Resample S2 band to HS grid shape (nearest for demo — same extent assumed)
    s2_band = s2_arrays[band].ravel()
    n = min(hs_band.size, s2_band.size)
    slope, intercept = fit_linear_calibration(hs_band[:n], s2_band[:n])
    calibration[band] = (slope, intercept)
    print(f'{band}: HS = {slope:.3f} * S2 + {intercept:.5f}')

# Apply calibration and recompute indices
b03_cal = apply_calibration(s2_arrays['B03'], *calibration['B03'])
b04_cal = apply_calibration(s2_arrays['B04'], *calibration['B04'])
b05_cal = apply_calibration(s2_arrays['B05'], *calibration['B05'])

ndci_s2_cal = compute_s2_ndci(b04_cal, b05_cal)
doc_s2_cal = compute_s2_doc_proxy(b03_cal, b04_cal)
turb_s2_cal = compute_s2_turbidity_proxy(b03_cal, b04_cal)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes, [ndci_s2_cal, doc_s2_cal, turb_s2_cal],
                          ['Calibrated NDCI', 'Calibrated DOC', 'Calibrated Turbidity']):
    im = ax.imshow(data, cmap='RdYlGn')
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Sentinel-2 indices after airborne calibration')
plt.tight_layout()
plt.show()

In [ ]:
from hs_utils import convolve_to_s2

# SAM on Sentinel-2 using library spectra convolved to S2 bands
s2_ref = {}
for cls, ref in ref_spectra.items():
    conv = convolve_to_s2(ref, wavelengths)
    s2_ref[cls] = np.array([conv[b] for b in ['B03', 'B04', 'B05', 'B08', 'B11']])

s2_stack = np.stack([s2_arrays[b] for b in ['B03', 'B04', 'B05', 'B08', 'B11']], axis=-1)
s2_flat = s2_stack.reshape(-1, 5)

sam_s2 = {}
for cls, ref_vec in s2_ref.items():
    sam_s2[cls] = spectral_angle(s2_flat, ref_vec).reshape(s2_stack.shape[:2])

stack_s2 = np.stack(list(sam_s2.values()), axis=-1)
label_idx_s2 = np.argmin(stack_s2, axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(label_idx, cmap='tab10', vmin=0, vmax=len(class_names) - 1)
axes[0].set_title('SAM — airborne')
axes[0].axis('off')
axes[1].imshow(label_idx_s2, cmap='tab10', vmin=0, vmax=len(class_names) - 1)
axes[1].set_title('SAM — Sentinel-2 (calibrated library)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print('Done.')
